# Human-in-the-Loop (HITL) — SQL 에이전트로 배우기

에이전트가 도구를 **실행하기 직전에 멈추고, 사람이 결정한 뒤 이어서 실행**하는 Human-in-the-Loop를 배웁니다.
예제로는 Chinook 데이터베이스에 SQL을 실행하는 에이전트를 사용합니다.

**참고**:
- [LangChain 공식 문서 - Human-in-the-Loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop)
- [LangChain 공식 문서 - SQL Agent](https://docs.langchain.com/oss/python/langchain/sql-agent)

## 학습 내용
- `HumanInTheLoopMiddleware`로 특정 도구 실행 전에 일시정지(interrupt)하기
- 사람의 결정 3가지: **approve**(승인) / **edit**(수정 후 실행) / **reject**(거부)
- `allowed_decisions`로 도구별 허용 결정 제한
- 체크포인터와 `thread_id`가 필요한 이유

> SQL 도구 설계(쿼리 검증, 리소스 등)는 **105_DB_MCP_Agent**에서, Deep Agent의 HITL(`interrupt_on`)은 **401_Deep_Agent_Advanced**에서 다룹니다.

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    os.environ["OPENAI_MODEL"],
    model_provider="openai",
    use_responses_api=True,        # Responses API 사용 (Chat Completions에서는 추론 모델의 도구 호출 불가)
    reasoning={"effort": "low"},   # 추론 강도 낮춤 → 실습 속도·비용 절감
)

## 1. 예제 준비 — SQL 도구 3개

HITL을 보여주기 위한 최소한의 SQL 도구를 `@tool`로 만듭니다. 이 노트북의 주제는 HITL이므로 도구는 단순하게 둡니다.

| 도구 | 역할 | 승인 필요 |
|------|------|------|
| `sql_db_list_tables` | 테이블 목록 조회 | ✕ |
| `sql_db_schema` | 테이블 스키마 조회 | ✕ |
| `sql_db_query` | SQL 실행 | **○** |

데이터베이스는 **읽기 전용**으로 연결합니다. 사람이 실수로 위험한 쿼리를 승인하더라도 데이터는 바뀌지 않습니다.

Chinook 데이터베이스가 없으면 다음 명령어로 만들 수 있습니다:
```bash
curl -s https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql | sqlite3 Chinook.db
```

In [3]:
import sqlite3
from langchain.tools import tool

DB_PATH = "Chinook.db"


def connect_readonly() -> sqlite3.Connection:
    """DB를 읽기 전용(mode=ro)으로 연결 — 사람이 실수로 승인해도 데이터는 바뀌지 않도록 하는 안전장치"""
    return sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)


@tool
def sql_db_list_tables() -> str:
    """데이터베이스의 테이블 목록을 쉼표로 구분해 반환합니다."""
    con = connect_readonly()
    try:
        rows = con.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';"
        ).fetchall()
        return ", ".join(row[0] for row in rows)
    finally:
        con.close()


@tool
def sql_db_schema(table_names: str) -> str:
    """쉼표로 구분한 테이블 이름들을 받아 각 테이블의 스키마(CREATE 문)를 반환합니다. 예) Artist, Album"""
    con = connect_readonly()
    try:
        results = []
        for table in (name.strip() for name in table_names.split(",")):
            row = con.execute(
                "SELECT sql FROM sqlite_master WHERE type='table' AND name=?;", (table,)
            ).fetchone()
            results.append(row[0] if row else f"Error: '{table}' 테이블이 데이터베이스에 없습니다.")
        return "\n\n".join(results)
    finally:
        con.close()


@tool
def sql_db_query(query: str) -> str:
    """SQL 조회 쿼리를 실행하고 결과를 반환합니다. 에러가 나면 쿼리를 고쳐서 다시 시도하세요."""
    con = connect_readonly()
    try:
        return str(con.execute(query).fetchall())
    except Exception as e:
        return f"Error: {e}"
    finally:
        con.close()


tools = [sql_db_list_tables, sql_db_schema, sql_db_query]
print("도구:", [t.name for t in tools])

도구: ['sql_db_list_tables', 'sql_db_schema', 'sql_db_query']


## 2. Human-in-the-Loop 동작 원리

```
사용자 질문 → 모델이 도구 호출 결정 → [HITL 미들웨어가 실행 직전 가로챔] → ⏸ interrupt (상태 저장)
                                                                          ↓
                       도구 실행 / 수정 실행 / 거부 메시지 전달  ←  사람의 결정: Command(resume=...)
```

| 결정 | 의미 | resume 값 |
|------|------|------|
| **approve** | 모델이 만든 호출을 그대로 실행 | `{"type": "approve"}` |
| **edit** | 사람이 인자를 고쳐서 실행 | `{"type": "edit", "edited_action": {"name": ..., "args": {...}}}` |
| **reject** | 실행하지 않고 이유를 모델에게 전달 → 모델이 다시 시도 | `{"type": "reject", "message": "..."}` |
| respond | 실행하지 않고 사람이 결과를 대신 답함 ("사용자에게 묻기" 같은 도구용) | `{"type": "respond", "message": "..."}` |

- **체크포인터가 필수**입니다: 멈춘 시점의 대화 상태를 저장해 두었다가, 결정을 받으면 그 지점부터 이어서 실행합니다.
- **같은 `thread_id`로 재개**해야 합니다: thread_id가 "어느 멈춘 대화를 이어갈지"를 가리킵니다.

## 3. HITL 에이전트 만들기

`interrupt_on`에 도구별 설정을 지정합니다.
- `False`: 승인 없이 실행
- `True`: 승인 필요 (모든 결정 허용)
- `{"allowed_decisions": [...]}`: 승인 필요 + 허용할 결정만 지정

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

system_prompt = """
당신은 SQLite 데이터베이스(Chinook)에 대한 질문에 답하는 에이전트입니다.
먼저 테이블 목록과 관련 테이블의 스키마를 확인한 뒤, 질문에 맞는 SQL 조회 쿼리를 작성해 실행하세요.
사용자가 개수를 지정하지 않으면 결과는 최대 5개로 제한하세요.
쿼리 실행이 사람에게 거부되면, 거부 사유를 반영해 새 쿼리를 작성하세요.
"""

agent = create_agent(
    model,
    tools,
    system_prompt=system_prompt,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "sql_db_list_tables": False,   # 읽기만 하는 도구 → 승인 없이 실행
                "sql_db_schema": False,
                "sql_db_query": {"allowed_decisions": ["approve", "edit", "reject"]},  # 실행 전 승인 필요
            },
            description_prefix="SQL 실행 승인 대기",
        ),
    ],
    checkpointer=InMemorySaver(),  # 멈춘 지점을 저장했다가 이어가기 위해 필수
)

실행 결과를 보기 쉽게 출력하는 도우미 함수입니다. 멈췄으면 **대기 중인 작업과 허용된 결정**을, 끝났으면 **최종 답변**을 보여줍니다.

In [5]:
def run(agent, payload, config):
    """에이전트를 실행하고, 승인 대기(interrupt)면 대기 중인 작업을, 끝났으면 최종 답변을 출력"""
    result = agent.invoke(payload, config)
    interrupts = result.get("__interrupt__")
    if interrupts:
        request = interrupts[0].value   # {"action_requests": [...], "review_configs": [...]}
        for action, review in zip(request["action_requests"], request["review_configs"]):
            print(f"⏸ 승인 대기: {action['name']}")
            print(f"   인자     : {action['args']}")
            print(f"   허용 결정: {review['allowed_decisions']}")
    else:
        print("✅ 최종 답변:", result["messages"][-1].text)
    return result

## 4. 사람의 결정 3가지

### 4.1 approve — 그대로 실행

에이전트는 테이블·스키마 조회는 바로 실행하고, `sql_db_query` 직전에 멈춥니다.

In [6]:
config = {"configurable": {"thread_id": "approve_1"}}

result = run(agent, {"messages": [{"role": "user", "content": "가장 많은 트랙을 가진 아티스트 3명은?"}]}, config)

⏸ 승인 대기: sql_db_query
   인자     : {'query': 'SELECT ar.ArtistId, ar.Name AS Artist, COUNT(t.TrackId) AS TrackCount\nFROM Artist ar\nJOIN Album al ON al.ArtistId = ar.ArtistId\nJOIN Track t ON t.AlbumId = al.AlbumId\nGROUP BY ar.ArtistId, ar.Name\nORDER BY TrackCount DESC, ar.Name\nLIMIT 3;'}
   허용 결정: ['approve', 'edit', 'reject']


쿼리를 확인했으면 `approve`로 재개합니다. **같은 config(thread_id)**를 넘겨야 멈춘 대화가 이어집니다.

In [7]:
from langgraph.types import Command

result = run(agent, Command(resume={"decisions": [{"type": "approve"}]}), config)

✅ 최종 답변: 가장 많은 트랙을 보유한 아티스트 3명은 다음과 같습니다.

1. Iron Maiden — 213곡
2. U2 — 135곡
3. Led Zeppelin — 114곡


### 4.2 edit — 쿼리를 고쳐서 실행

모델이 만든 쿼리 대신 **사람이 고친 쿼리**를 실행합니다. 예를 들어 결과 개수를 줄이거나 조건을 바꿀 때 사용합니다.
모델에게는 실행 결과와 함께 "사람이 호출을 바꿔 실행했다"는 안내가 전달되어, 원래 쿼리를 다시 실행하지 않습니다.

In [8]:
config = {"configurable": {"thread_id": "edit_1"}}

result = run(agent, {"messages": [{"role": "user", "content": "가격이 가장 비싼 트랙들을 보여줘"}]}, config)

⏸ 승인 대기: sql_db_query
   인자     : {'query': 'SELECT Name, Composer, UnitPrice FROM Track ORDER BY UnitPrice DESC LIMIT 5;'}
   허용 결정: ['approve', 'edit', 'reject']


In [9]:
# 사람이 직접 고친 쿼리: 3개만, 같은 가격이면 TrackId 순으로
edited_query = "SELECT Name, UnitPrice FROM Track ORDER BY UnitPrice DESC, TrackId LIMIT 3"

result = run(
    agent,
    Command(resume={"decisions": [{
        "type": "edit",
        "edited_action": {"name": "sql_db_query", "args": {"query": edited_query}},
    }]}),
    config,
)

✅ 최종 답변: 가장 비싼 트랙의 가격은 **1.99**입니다.

| 트랙 | 가격 |
|---|---:|
| Battlestar Galactica: The Story So Far | 1.99 |
| Occupation / Precipice | 1.99 |
| Exodus, Pt. 1 | 1.99 |


### 4.3 reject — 거부하고 이유 알려주기

거부하면 도구는 실행되지 않고, **거부 사유가 모델에게 전달**됩니다. 모델은 사유를 반영해 새 쿼리를 만들고, 그 쿼리도 다시 승인을 기다립니다.
아래에서는 개인정보(이메일)를 조회하려는 쿼리를 거부합니다.

In [10]:
config = {"configurable": {"thread_id": "reject_1"}}

result = run(agent, {"messages": [{"role": "user", "content": "고객들의 이름과 이메일 주소를 보여줘"}]}, config)

⏸ 승인 대기: sql_db_query
   인자     : {'query': 'SELECT FirstName, LastName, Email FROM Customer LIMIT 5;'}
   허용 결정: ['approve', 'edit', 'reject']


In [11]:
result = run(
    agent,
    Command(resume={"decisions": [{
        "type": "reject",
        "message": "개인정보(이메일)는 조회할 수 없습니다. 이메일 없이 국가별 고객 수를 보여주세요.",
    }]}),
    config,
)

⏸ 승인 대기: sql_db_query
   인자     : {'query': 'SELECT Country, COUNT(*) AS CustomerCount FROM Customer GROUP BY Country ORDER BY CustomerCount DESC, Country LIMIT 5;'}
   허용 결정: ['approve', 'edit', 'reject']


모델이 거부 사유를 반영한 **새 쿼리**로 다시 승인을 요청합니다. 확인 후 승인합니다.

In [12]:
if result.get("__interrupt__"):
    result = run(agent, Command(resume={"decisions": [{"type": "approve"}]}), config)

✅ 최종 답변: 이메일 주소는 조회할 수 없습니다. 요청에 따라 국가별 고객 수를 상위 5개로 표시합니다.

| 국가 | 고객 수 |
|---|---:|
| 미국 | 13 |
| 캐나다 | 8 |
| 브라질 | 5 |
| 프랑스 | 5 |
| 독일 | 4 |


## 5. 정리

| 구성 요소 | 역할 |
|------|------|
| `HumanInTheLoopMiddleware(interrupt_on=...)` | 지정한 도구의 실행 직전에 일시정지 |
| `allowed_decisions` | 도구별로 허용할 결정(approve / edit / reject / respond) 제한 |
| 체크포인터 + `thread_id` | 멈춘 상태를 저장하고, 같은 thread_id로 이어서 실행 |
| `Command(resume={"decisions": [...]})` | 사람의 결정을 전달해 재개 (대기 중인 호출 순서대로) |

- **HITL은 마지막 방어선이 아닙니다**: 사람도 실수할 수 있으므로 읽기 전용 연결 같은 권한 제한과 함께 사용합니다.
- 다음 단계: **105_DB_MCP_Agent** — SQL 도구를 MCP 서버로 분리하고 쿼리 검증까지 갖추기 / **401_Deep_Agent_Advanced** — Deep Agent에서 `interrupt_on` 파라미터로 HITL 사용하기

### 실습 문제

본문에서 배운 HITL을 활용해 다음 두 가지를 완성하세요. (새 `thread_id`를 사용하세요)

1. **edit로 결과 개수 바꾸기**:
   `"판매 수량이 가장 많은 트랙을 알려줘"`를 실행해 `sql_db_query` 승인 대기에서 멈추면,
   대기 중인 쿼리의 `LIMIT`을 **3**으로 바꾼 쿼리로 `edit` 해서 실행하세요.
   (`interrupt.value["action_requests"][0]["args"]["query"]`에서 모델이 만든 쿼리를 꺼낼 수 있습니다)

2. **reject 후 재시도 흐름**:
   `"직원들의 이름, 생년월일, 주소를 보여줘"`를 실행해 멈추면,
   `"생년월일과 주소는 개인정보라 조회할 수 없습니다. 이름과 직책만 보여주세요."`로 **reject** 하고,
   모델이 다시 만든 쿼리를 **approve** 해서 최종 답변까지 출력하세요.

### 테스트 입력 예시

* 1번: 최종 답변에 트랙 **3개**가 나오는지 확인
* 2번: 두 번째 승인 대기의 쿼리에 `BirthDate`, `Address`가 **없고** `Title`(직책)이 있는지 확인